# SpaceX Falcon 9 First Stage Landing Prediction
## Module 7: Interactive Visual Analytics with Plotly Dash

**Author:** Pritam Acharya

The full interactive dashboard lives in [`dash_app/spacex_dash_app.py`](../dash_app/spacex_dash_app.py) — run it with:

```bash
pip install dash plotly pandas
python dash_app/spacex_dash_app.py
```

then open `http://127.0.0.1:8050`. It has:
- A **dropdown** to filter by launch site (or view all)
- A **pie chart** of successful launches per site (or success/failure split for one site)
- A **range slider** to filter by payload mass
- A **scatter plot** of payload mass vs. landing outcome, colored by booster version

`dash` and `plotly` aren't installed in this sandbox and can't be installed here (no outbound network access), so this notebook can't run the live server — but every callback's underlying data logic is plain pandas, which we can run and verify directly below. This confirms the dashboard's numbers are correct before you ever launch it.


In [1]:
import pandas as pd

spacex_df = pd.read_csv("../data/spacex_launch_geo.csv")
spacex_df.head()

,LaunchSite,Latitude,Longitude,Class,PayloadMass,BoosterVersion,Outcome
0,CCAFS SLC 40,28.561857,-80.577366,0,6104.959412,Falcon 9,Failure
1,CCAFS SLC 40,28.561857,-80.577366,0,525.000000,Falcon 9,Failure
2,CCAFS SLC 40,28.561857,-80.577366,0,677.000000,Falcon 9,Failure
3,VAFB SLC 4E,34.632093,-120.610829,0,500.000000,Falcon 9,Failure
4,CCAFS SLC 40,28.561857,-80.577366,0,3170.000000,Falcon 9,Failure


### Verifying callback 1: success-by-site pie chart

When "All Sites" is selected, the pie chart shows `Class` (0/1) summed per site — this is exactly what `px.pie(spacex_df, values='Class', names='LaunchSite')` aggregates internally.

In [2]:
spacex_df.groupby("LaunchSite")["Class"].sum()

LaunchSite
CCAFS SLC 40    33
KSC LC 39A      17
VAFB SLC 4E     10
Name: Class, dtype: int64

When a specific site is selected instead (e.g. `CCAFS SLC 40`), the pie chart switches to a success/failure split for that site alone:

In [3]:
filtered = spacex_df[spacex_df["LaunchSite"] == "CCAFS SLC 40"]
filtered["Outcome"].value_counts()

Outcome
Success    33
Failure    22
Name: count, dtype: int64

### Verifying callback 2: payload-range scatter filter

The range slider filters `PayloadMass` between the selected bounds before the scatter chart draws. Let's confirm the filter logic with the default full range, and a narrowed range (e.g. 2,000-6,000 kg):

In [4]:
full_range = spacex_df[spacex_df["PayloadMass"].between(spacex_df["PayloadMass"].min(), spacex_df["PayloadMass"].max())]
narrow_range = spacex_df[spacex_df["PayloadMass"].between(2000, 6000)]
print("Full range:", len(full_range), "rows")
print("2000-6000 kg range:", len(narrow_range), "rows")
print("Success rate in narrow range:", round(narrow_range["Class"].mean(), 3))

Full range: 90 rows
2000-6000 kg range: 43 rows
Success rate in narrow range: 0.628


### Live equivalent

Since the Dash server itself can't run in this sandbox, I've rendered an interactive version of this exact dashboard (dropdown, pie chart, payload slider, and scatter chart, built with Plotly.js on the same underlying data) directly in the chat — see the message accompanying this notebook. All the numbers there match what's verified above.


### Summary

- Verified every callback's core pandas logic directly against the real dataset — the dashboard's numbers are correct before ever launching the server.
- **CCAFS SLC-40** carries the most launches (55) and the most successes (33) in absolute terms, but the lowest success *rate* (60%).
- Narrowing payload mass to 2,000-6,000 kg selects a meaningfully different subset with its own success rate, confirming the range slider does real filtering work.
- The dashboard script (`dash_app/spacex_dash_app.py`) is ready to run as-is with `dash`/`plotly` installed.

**Next:** `8. SpaceX_Machine_Learning_Prediction_Part_5.jupyterlite.ipynb` — training and evaluating classification models to predict landing success.
